# Analisis Hasil Training Q-Learning

Notebook ini membaca hasil **Go** yang sudah tersimpan. Tidak ada unduhan OSM, training, pembaruan Q, atau penghitungan ulang reward di Python.

Alurnya mengikuti notebook A/B/C proyek acuan: evaluasi per seed, agregat profil, grafik training, diagnosis, overlap, dan perbandingan konfigurasi. Semua seed, termasuk rute gagal, tetap dihitung. Peta interaktif tetap berasal dari HTML Go.

Pilih **Restart Kernel and Run All** setelah hasil Go diperbarui. Angka di sini mengikuti metadata hasil tersimpan, bukan perubahan YAML setelah training.

Implementasi analisis dan tampilan berada di `analysis/training.py`. Ubah parameter pada cell awal, lalu jalankan bagian laporan yang diperlukan.


## 1. Pilihan data

Ubah `WORKDIR` untuk A, B, atau C. `PROFILES=None` membaca semua profil yang sudah selesai; isi daftar nama untuk membatasi analisis. Profil yang belum tersedia ditampilkan pada tabel ketersediaan dan tidak dianggap gagal training.

`ROLLING_WINDOW=50` mengikuti acuan. `PLOT_EVERY=1` menggambar seluruh episode seperti acuan. Nilai lebih besar dapat dipakai untuk pratinjau ringan; pengurangan titik dilakukan **setelah** rata-rata bergerak dihitung. Ringkasan akhir memakai seluruh observasi pada 10.000 episode terakhir, atau seluruh episode untuk uji yang lebih pendek.


In [ ]:
import importlib
from analysis import common, training

importlib.reload(common)
importlib.reload(training)
ROOT = training.PROJECT_ROOT

# Edit these settings; implementation lives in analysis/training.py.
WORKDIR = ROOT / "workdirs" / "scenario-b"
PROFILES = None
ROLLING_WINDOW = 50
TAIL_EPISODES = 10_000
PLOT_EVERY = 1
CHUNK_SIZE = 50_000

COMPARE_WORKDIRS = [ROOT / "workdirs" / f"scenario-{name}" for name in ("a", "b", "c")]
SERVER_ROOT = ROOT / "workdirs"
SERVER_BASE = "http://127.0.0.1:8080"

training.show_environment(WORKDIR)

## 2. Memuat metadata dan evaluasi greedy

Metadata harus berstatus `completed`. Bila percobaan terbaru suatu profil gagal atau masih berjalan, notebook menolak memakai hasil lama yang tertinggal tanpa penjelasan. Seed dibaca sebagai teks agar angka `uint64` tidak kehilangan presisi.

`is_loop` berarti kembali ke start dalam toleransi jarak. `closing_action_available` hanya menunjukkan kesempatan menutup loop. Comfort adalah skor preferensi berbobot panjang, bukan persentase kenyamanan atau ukuran keamanan.


In [ ]:
report = training.TrainingAnalysis(
    WORKDIR, profiles=PROFILES, rolling_window=ROLLING_WINDOW,
    tail_episodes=TAIL_EPISODES, plot_every=PLOT_EVERY, chunk_size=CHUNK_SIZE,
)
report.load()
report.show_configurations()

## 3. Membaca history secara bertahap

CSV dibaca per 50.000 baris agar seluruh history jutaan episode tidak ditahan di RAM. Jumlah episode, seed, urutan episode, dan epsilon diperiksa terhadap metadata.

Kurva utama merata-ratakan seed per episode, lalu memakai rata-rata bergerak 50 episode. Khusus overlap, jumlah dan cacah observasi digulir bersama agar setiap observasi memiliki bobot yang sama. Nilai `NaN` tetap berarti tidak ada observasi, bukan nol.


## 4. Hasil per konfigurasi, mengikuti notebook acuan

Setiap profil menampilkan tabel metrik vertikal dengan seed sebagai kolom,
ringkasan diagnosis seluruh seed, termination rate training/evaluasi, lalu grafik
training **3 × 2**, diagnosis **lima panel**, dan overlap **satu panel**.
Ukuran gambar, urutan panel, warna, judul, label, dan legenda mengikuti
`scenario_A/B/C.ipynb` dan fungsi grafik di `rute_lari_core.py` pada proyek acuan.

Semua episode digambar (`PLOT_EVERY=1`). Grafik memakai jendela 50 episode;
epsilon tetap nilai asli. Sumbu comfort diperluas jika skor keluar dari 0–1 agar
nilai negatif/di luar rentang tetap terlihat. Seluruh seed berhasil/gagal tetap
dihitung. Statistik 10.000 episode terakhir tersedia sebagai CSV tambahan.

In [ ]:
report.show_profiles()

## 5. Ringkasan hasil tersimpan, mengikuti section 3 notebook acuan

Perbandingan dipisahkan menjadi **epsilon dinamis** dan **horizon epsilon tetap**.
Masing-masing memakai tabel seluruh seed serta grafik **2 × 2**: keberhasilan
loop greedy, galat jarak, konsistensi mendekati start, dan kesempatan closing.
Sumbu X adalah jumlah episode training; metrik proporsi tetap ditampilkan 0–1,
bukan diubah menjadi persen. Overlap evaluasi ditampilkan terpisah, satu panel
untuk setiap strategi epsilon, dalam persen dan mencakup seluruh seed.

Metadata harus sepadan untuk snapshot, start, seed, reward, mask, RNG, dan
hyperparameter. Profil yang belum selesai tidak diberi nilai nol. Ringkasan
lengkap tetap diekspor ke `profile_summary.csv`.

In [ ]:
report.compare_profiles()

## 6. Perbandingan A/B/C bila hasilnya tersedia

Bagian ini membaca evaluasi saja. Data yang belum ada dilaporkan dan tidak diisi dengan angka nol. Grafik perbandingan state baru dibuat bila minimal dua scenario memiliki profil, durasi, seed, graf, reward, dan parameter bersama yang sama. Scenario yang berbeda menjadi faktor pembanding.


In [ ]:
report.compare_scenarios(COMPARE_WORKDIRS)

## 7. Peta dan interpretasi hasil

Jalankan `go run ./cmd/cli serve --dir workdirs --port 8080`, lalu buka tautan peta berikut. HTML dibuat oleh Go; notebook tidak membangkitkan ulang rute. Peta tidak disematkan otomatis agar analisis tabel/grafik tetap bisa berjalan tanpa koneksi tile.

Seluruh tabel/diagram turunan disimpan di `output_dir/analysis/training/`; file training asli tidak diubah. Graf Go masih mentah dan jarak ke start memakai Haversine, sehingga angka tidak dijanjikan identik dengan eksperimen Python yang memakai graf terproyeksi/disimplifikasi. Hasil ini belum merupakan validasi lapangan.


In [ ]:
report.show_maps(SERVER_ROOT, SERVER_BASE)